### Create DB if file does not exists and add data_import_control Table

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
from datetime import datetime
import re
from datetime import datetime


# config

photometer_name = 'stars930'
db_name = f'../data/tess_data.db'
ecsv_folder = f'TESS-IDA-TOOLS/jupyter/ECSV/{photometer_name}'

# SQLite-Verbindung erstellen
engine = create_engine(f'sqlite:///{db_name}')

# Tabelle in SQLite erstellen (falls nicht vorhanden)
def execute_query(query):
    with engine.connect() as connection:
        result = connection.execute(text(query))
        return result.fetchall()

def execute_command(command):
    with engine.connect() as connection:
        connection.execute(text(command))
        connection.commit()


# Example usage:
command = """
CREATE TABLE IF NOT EXISTS data_import_control (
    name TEXT NOT NULL,
    date_of_data_name TIMESTAMP NOT NULL,
    date_of_import TIMESTAMP NOT NULL,
    complete BOOLEAN NOT NULL
);
"""

execute_command(command)




### Create Photometer Table if it does not exists 

In [14]:
query = f"""SELECT EXISTS (SELECT 1 FROM sqlite_master WHERE type='table' AND name='{photometer_name}_data') AS table_exists;"""
result = execute_query(query)
print

if result == [(0,)]:
    create_table_photometer_query = f'''
    CREATE TABLE {photometer_name}_data (
        time TIMESTAMP,
        enclosure_temperature DOUBLE PRECISION,
        sky_temperature DOUBLE PRECISION,
        frequency DOUBLE PRECISION,
        msas DOUBLE PRECISION,
        zp DOUBLE PRECISION,
        sequence_number BIGINT,
        sun_alt DOUBLE PRECISION,
        moon_alt DOUBLE PRECISION,
        moon_illumination DOUBLE PRECISION
    );
    '''

    # create table
    with engine.connect() as connection:
        connection.execute(text(create_table_photometer_query))
    print("created table")
else:
    print("exists")

created table


### Photometer Table

In [15]:
command = f"DROP TABLE IF EXISTS {photometer_name}_data"
execute_command(command)


### Create Photometer Table

In [16]:
create_table_photometer_query = f'''
CREATE TABLE {photometer_name}_data (
    time TIMESTAMP,
    enclosure_temperature DOUBLE PRECISION,
    sky_temperature DOUBLE PRECISION,
    frequency DOUBLE PRECISION,
    msas DOUBLE PRECISION,
    zp DOUBLE PRECISION,
    sequence_number BIGINT,
    sun_alt DOUBLE PRECISION,
    moon_alt DOUBLE PRECISION,
    moon_illumination DOUBLE PRECISION
);
'''

# create table
with engine.connect() as connection:
    connection.execute(text(create_table_photometer_query))

### Create entry in the data_import_control for the file

In [6]:
import datetime

# Set the data variable
data = f"{photometer_name}_2024-02"
data = data.split("_")[1].replace("-", "")  # Extract date portion and remove the hyphen

# Convert the extracted data to a datetime object and set to first day of the month
date_of_data_name = datetime.datetime.strptime(data, "%Y%m").date().replace(day=1)  # Correct format for date_of_data_name

# Current datetime for date_of_import
date_of_import = datetime.datetime.now().date()

# Calculate the first day of the next month after date_of_data_name
next_month = (date_of_data_name.replace(day=1) + datetime.timedelta(days=31)).replace(day=1)

# Insert command with updated conditional `complete` value
command = f"""
INSERT INTO data_import_control (name, date_of_data_name, date_of_import, complete) 
VALUES (
    '{photometer_name}',
    '{date_of_data_name}',
    '{date_of_import}',
    CASE WHEN date('{date_of_import}') >= date('{next_month}') THEN 1 ELSE 0 END
);
"""
# Execute the command
execute_command(command)


### Test if there need to get new data

-> Falsche Handhabung: Startmonat wird auch in Liste gespeichert, obwohl Eintrag in Data import control


In [7]:
# Generate a list of months between start and end dates
from datetime import datetime, timedelta

start_date = datetime.strptime("2024-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2024-11-04", "%Y-%m-%d")
date_list = []


# Populate date_list with the first of each month in the range
current_date = start_date
while current_date <= end_date:
    date_list.append(current_date.strftime("%Y-%m-%d"))
    # Move to the first of the next month
    next_month = current_date.month % 12 + 1
    next_year = current_date.year + (current_date.month // 12)
    current_date = current_date.replace(year=next_year, month=next_month, day=1)

# Fetch existing dates in the database
query = f"""
SELECT date_of_data_name
FROM data_import_control
WHERE name = '{photometer_name}'
AND complete = 1
AND date_of_data_name BETWEEN '{start_date}' AND '{end_date}'
"""

# Execute the query and get results
existing_dates = execute_query(query)

# Extract dates from the query result as a set
existing_dates = {row[0] for row in existing_dates}

# Remove existing dates from date_list
filtered_dates = [date for date in date_list if date not in existing_dates]

# Print filtered dates
required_month_files = [date[:7] for date in filtered_dates]  # Extracts "YYYY-MM" from each date
print(required_month_files)



['2024-01', '2024-10', '2024-11']


### Download required files from list

So i need to install the TESS-IDA lib

In [44]:
import os

# List of dates
# dates = ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07']

dates = required_month_files

# Iterate through each date and run the command
for date in dates:
    command = f"tess-ida-pipe --console single --in-dir IDA --out-dir ECSV --name {photometer_name} --month {date}"
    print(f"Running command: {command}")
    os.system(command)  # Executes the command in the shell


Running command: tess-ida-pipe --console single --in-dir IDA --out-dir ECSV --name stars926 --month 2024-01
Running command: tess-ida-pipe --console single --in-dir IDA --out-dir ECSV --name stars926 --month 2024-02
Running command: tess-ida-pipe --console single --in-dir IDA --out-dir ECSV --name stars926 --month 2024-11


In [ ]:
import subprocess
import os

# Set the relative path for the command directory and activation script
command_dir = os.path.join("python", "TESS-IDA-TOOLS", "jupyter")
activate_script = os.path.join(command_dir, ".venv", "Scripts", "Activate.ps1")  # Correct path to activate.ps1

# Define the list of months (ensure `required_month_files` is set)
dates = required_month_files

# Iterate through each date and run the command in the environment
for date in dates:
    # Form the tess-ida-pipe command
    command = f"tess-ida-pipe --console single --in-dir IDA --out-dir ECSV --name {photometer_name} --month {date}"
    print(f"Running command for date {date}: {command}")

    # Full command to activate the environment and run the tess-ida-pipe command
    full_command = f"powershell -NoExit -Command \"cd {command_dir}; . '{activate_script}'; {command}\""

    # Run the command and capture both stdout and stderr
    result = subprocess.run(full_command, capture_output=True, text=True, shell=True)

    # Check if the command was successful and capture both stdout and stderr
    if result.returncode == 0:
        print(f"Command for {date} completed successfully.")
        print("Output:\n", result.stdout)  # Show the standard output
    else:
        print(f"Command for {date} failed with return code {result.returncode}.")
        print("Error output:\n", result.stderr)  # Show any error output




### Importing Data to Database (without Filter)

In [30]:
# # Muster zur Überprüfung des Dateinamensformats
filename_pattern = re.compile(rf"{photometer_name}_\d{{4}}-\d{{2}}\.ecsv")

# Importieren und Filtern der ECSV-Daten
for file_name in os.listdir(ecsv_folder):
    # Überprüfen, ob der Dateiname auf ".ecsv" endet und dem Format entspricht
    if file_name.endswith('.ecsv') and filename_pattern.match(file_name):
        ecsv_file = os.path.join(ecsv_folder, file_name)
        data = pd.read_csv(ecsv_file, comment='#', delimiter=',')

        # Spaltennamen anpassen
        data.columns = [
            'time', 
            'enclosure_temperature', 
            'sky_temperature', 
            'frequency', 
            'msas', 
            'zp', 
            'sequence_number', 
            'sun_alt', 
            'moon_alt', 
            'moon_illumination'
        ]

        # Gefilterte Daten in die SQLite-Datenbank importieren
        data.to_sql(f'{photometer_name}_data', engine, if_exists='append', index=False)
        print(f"Gefilterte Daten von {file_name} wurden erfolgreich importiert.")
    else:
        print(f"Datei {file_name} entspricht nicht dem erwarteten Format und wird übersprungen.")

print("Alle gefilterten Dateien wurden in die SQLite-Datenbank importiert.")


Gefilterte Daten von stars926_2023-10.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars926_2023-11.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars926_2023-12.ecsv wurden erfolgreich importiert.
Datei stars926_202310-202409.ecsv entspricht nicht dem erwarteten Format und wird übersprungen.
Gefilterte Daten von stars926_2024-02.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars926_2024-03.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars926_2024-04.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars926_2024-05.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars926_2024-06.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars926_2024-07.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars926_2024-08.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars926_2024-09.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars926_2024-10.ecsv wurden erfolgreich importiert.
Gefilterte Daten

### Importing Data to Database (with Filter - local Time 17:00 until 08:00)

In [17]:
import os
import re
import pandas as pd
from datetime import datetime, timedelta

# Muster zur Überprüfung des Dateinamensformats
filename_pattern = re.compile(rf"{photometer_name}_\d{{4}}-\d{{2}}\.ecsv")

# Importieren und Filtern der ECSV-Daten
for file_name in os.listdir(ecsv_folder):
    # Überprüfen, ob der Dateiname auf ".ecsv" endet und dem Format entspricht
    if file_name.endswith('.ecsv') and filename_pattern.match(file_name):
        ecsv_file = os.path.join(ecsv_folder, file_name)
        data = pd.read_csv(ecsv_file, comment='#', delimiter=',')
        
        # Spaltennamen anpassen
        data.columns = [
            'time', 
            'enclosure_temperature', 
            'sky_temperature', 
            'frequency', 
            'msas', 
            'zp', 
            'sequence_number', 
            'sun_alt', 
            'moon_alt', 
            'moon_illumination'
        ]
        
        # Konvertiere Zeitspalte in datetime und verschiebe um 2 Stunden auf Lokalzeit
        data['time'] = pd.to_datetime(data['time']) + timedelta(hours=2)
        
        # Filter für die Nachtstunden (17:00 bis 08:00)
        data = data[(data['time'].dt.hour >= 17) | (data['time'].dt.hour < 8)]
        
        # Gefilterte Daten in die SQLite-Datenbank importieren
        data.to_sql(f'{photometer_name}_data', engine, if_exists='append', index=False)
        print(f"Gefilterte Daten von {file_name} wurden erfolgreich importiert.")
    else:
        print(f"Datei {file_name} entspricht nicht dem erwarteten Format und wird übersprungen.")

print("Alle gefilterten Dateien wurden in die SQLite-Datenbank importiert.")

Gefilterte Daten von stars928_2024-10.ecsv wurden erfolgreich importiert.
Gefilterte Daten von stars928_2024-11.ecsv wurden erfolgreich importiert.
Datei stars928_202410-202411.ecsv entspricht nicht dem erwarteten Format und wird übersprungen.
Alle gefilterten Dateien wurden in die SQLite-Datenbank importiert.
